### Import Libraries & API Keys

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import json
import requests
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing")

### Set up Pushover

In [2]:
#step 2a -> Set up account in your browser
#step 2b -> Set up the app on your iPhone / Android, log into the same account
#step 2c -> In the browser createw an "Application/API Token"
#Step 2d -> Copy your User Key and API Token into the .env file,
#like this but with your own keys:
#PUSHOVER_USER=xxxxxxx
#PUSHOVER_TOKEN=yyyyyy

#Save changes to the .env file
#Run the test to manually send a notification from Pushover in your browser to your phone

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [4]:
#use this (!in private!) to test that your keys have been loaded

# print(pushover_user)
# print(pushover_token)

In [5]:
#Test Pushover
def send_notification(message: str):
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message
    }
    requests.post(pushover_url, data=payload)

In [6]:
# send_notification("Hello to myself from my AI engineering training")

### Step 3: Describe Pushover as an LLM tool

In [7]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events, completed tasks, or time-sensitive information",
    "parameters" : {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pushover to the list of tools for the LLM

In [8]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 5: Calling the tool from an LLM

In [ ]:
def handle_tool_calls(tool_calls):
    tool_call_results = []

    for tool_call in tool_calls:
        if tool_call.function.name == "send_notification":
            args = json.loads(tool_call.function.arguments) 
            message = args.get("message")
            send_notification(message)
            content = f"Notification sent: {message}"
        # elif tool_call.function.name == "function_name_2":
        #     content = 'function_name_2(args["message"])'
        else:
            content = f"Unknown tool call: {tool_call.function.name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
            
        tool_call_results.append(tool_call_result)
        
    return tool_call_results

In [ ]:
client = OpenAI()
messages = [{"role": "user", "content": "Please send me two notifications: 1( The amazing progress\
         I'm making on the AI Engineering training by SuperDataScience. 2) The fact that I am learning to use the OpenAI API and Pushover to send notifications to my phone!"}]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)


message = response.choices[0].message

#check if model wants to call a tool
if message.tool_calls:
    tool_results = handle_tool_calls(message.tool_calls)
    messages.append(message)

    for tool_result in tool_results:
        messages.append(tool_result)

    #you can also use messages.extend(tool_results)
    #* is the equivalent of unpacking the list into individual elements (javascript spread operator)
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        # tools=tools #TODO - consecutive tools calls will be added in the future
    )
    message = response.choices[0].message

print(message.content)

I've sent you two notifications:
1. The amazing progress you're making on the AI Engineering training by SuperDataScience.
2. The fact that you are learning to use the OpenAI API and Pushover to send notifications to your phone!
